This notebook gives you an overview of the module/framework. If you deconstract it the module has 4 important pieces:
- Three classes bronze, silver and gold (and a generic etl class)
- five actions load, transform, write, tblproperties and optimize
- On class and action lavel you can/have to pass different options as defined in Docstrings or here: https://nikkthegreek.codeberg.page/#options
- On class level you have multiple functions to overwrite as defined in docstrings or here: https://nikkthegreek.codeberg.page/#functions

Btw: All configs can also be passed as json or yaml config

**We start with defining our CATALOG and creating our schemas if they do not exist and importing our lakehouse classes bronze, silver, gold**

In [0]:
# replace with your catalog
CATALOG = spark.catalog.currentCatalog()
CATALOG = "nikkthegreek"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

In [0]:
from lakehouse.spark import bronze, silver, gold
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

# 1. Bronze

- We define our options as json (we can also directly pass them in the instance)
- We define our custom_load function. Here we just load the data as typical in bronze without any transformations
- We create an instance
- We load and write the data and execute it for the table name people

The module also runs operations like setting liquid to AUTO and some other best practice delta properties for you. You can of course configure this :) 

In [0]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [0]:
class NikksBronze(bronze.Bronze):
    def custom_load(self, table):
        return spark.read.table("samples.bakehouse.sales_transactions")

bronze_instance = NikksBronze(spark, **options)

In [0]:
bronze_instance.load().transform().write(mode="overwrite").execute("sales_transactions")

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.sales_transactions")
print(f"No. Rows: {df.count()}")
display(df)

In [0]:
df_history = spark.sql(f"DESCRIBE HISTORY {CATALOG}.bronze.sales_transactions")
display(df_history)

# 2 Silver

- Here we again define our options this time with a source schema
- We use here only our custom_transform functions with a simple date transformation. The load as handled automatically. You can of course also define custom load logics.
- And we run the execution again. This time with a load, transform and write 

In [0]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

In [0]:
class StarWarsSilver(silver.Silver):
    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = (
            df.withColumn("date", F.to_date("dateTime"))
            .withColumn("month", F.month("dateTime"))
            .withColumn("year", F.year("dateTime"))
        )

        return df

silver_instance = StarWarsSilver(spark, **options)

In [0]:
silver_instance.load().transform().write(mode="overwrite", merge_schema=True).execute(
    "sales_transactions"
)

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.sales_transactions")
print(f"No. Rows: {df.count()}")
display(df)

# 3 Gold

In [0]:
options = {
    "catalog": CATALOG,
    "source_schema": "silver",
    "target_schema": "gold",
}

In [0]:
class NikkGold(gold.Gold):
    def sales_per_month(self, df: DataFrame, table: str) -> DataFrame:
        return df.groupBy("month", "year").sum("totalPrice").withColumnRenamed("sum(totalPrice)", "totalPrice")

    def sales_per_product(self, df: DataFrame, table: str) -> DataFrame:
        return df.groupBy("product").sum("totalPrice").withColumnRenamed("sum(totalPrice)", "totalPrice")

gold_instance = NikkGold(spark, **options)

In [0]:
gold_instance.load(source_tbl="sales_transactions").transform(
    tbl_transformations={
        "salesPerMonth": "sales_per_month",
        "salesPerProduct": "sales_per_product",
    }
).write(mode="overwrite", merge_schema=True).execute("salesPerMonth", "salesPerProduct")

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.gold.salesPerMonth")
print(f"No. Rows: {df.count()}")
display(df)

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.gold.salesPerProduct")
print(f"No. Rows: {df.count()}")
display(df)

# 4 Debugging

An important requirement for users is to explore and debug intermediate steps as well as allowing unittesting.

Using the property data you can access the data for every execution step. First we check the loaded data by defining load. Secondly, we check the transformation.

You can also leverage the function pyspark.testing.assertDataFrameEqual to assert it to your expected dataframe

In [0]:
gold_instance.load(source_tbl="sales_transactions").execute("salesPerProduct")
display(gold_instance.data["salesPerProduct"])

In [0]:
gold_instance.load(source_tbl="sales_transactions").transform(
    tbl_transformations={
        "salesPerMonth": "sales_per_month",
        "salesPerProduct": "sales_per_product",
    }
).execute("salesPerProduct")
display(gold_instance.data["salesPerProduct"])

# 5 Tbl properties and optimize
- The tblproperties action allows you easily activating the most importan properties like type widening
- clusterby AUTO, CDF, deletion vectors, row tracking are automatically activated
- or any available property can be easily added as dict
- You can always run with the optimize action a vacuum, analyze or optimize

In [0]:
gold_instance.tblproperties(clusterby="AUTO", type_widening=True, tblproperties={"delta.deletedFileRetentionDuration":"interval 0 days"}).optimize(optimize=True, vacuum=True).execute("salesPerProduct")

In [0]:
df_history = spark.sql(f"DESCRIBE HISTORY {CATALOG}.gold.salesPerProduct")
display(df_history)

Clean Up

In [0]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.gold CASCADE")